In [49]:
!pip install import-ipynb

In [ ]:
import sys
import os
import torch
import torchvision.transforms as T
from PIL import Image
import requests
from io import BytesIO
from transformers import AutoTokenizer
from tqdm import tqdm
from google.colab import drive
drive.mount('DRIVE_PATH')
os.chdir('CLIP_MODEL_PATH')

import import_ipynb
from clip_model import CLIP

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIP().to(device)

model.load_state_dict(
    torch.load(
        "BEST_MODEL_PATH",
        map_location=device
    )
)

<All keys matched successfully>

In [52]:
captions = [
    # Animals
    "a red apple on white background",
    "a golden retriever dog sitting",
    "a white cat on a windowsill",
    "a horse standing in a field",
    "a rabbit in green grass",
    "a brown bear in the forest",
    "a penguin standing on ice",
    "a butterfly on a flower",
    # Food & Drink
    "a cup of black coffee",
    "a bowl of fresh strawberries",
    "a plate of sushi rolls",
    "a slice of chocolate cake",
    "a basket of croissants",
    "a glass of orange juice",
    "a bowl of ramen noodles",
    "a plate of colorful macarons",
    # Nature & Landscapes
    "snow covered mountain peak",
    "tropical beach at sunset",
    "a waterfall in a green forest",
    "a field of sunflowers",
]

tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

encoding = tokenizer(
    captions,
    padding="max_length",
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

In [53]:
inference_transform =T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711)
    )
])

In [54]:
image_urls = [
    "https://thumbs.dreamstime.com/b/red-apple-isolated-white-background-71475088.jpg",
    "https://media.istockphoto.com/id/652437872/photo/golden-retriever-adult-sitting-clowning-at-camera-isolated-on-white.jpg?s=612x612&w=0&k=20&c=VSrRacUEsMhTNZD5yqRzH0CfzE1VEdlIUey2RRgChCc=",
    "https://img.freepik.com/premium-photo/white-cat-is-relaxing-peacefully-windowsill_457211-16275.jpg",
    "https://images.pexels.com/photos/11882073/pexels-photo-11882073.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1",
    "https://previews.123rf.com/images/icefront/icefront1406/icefront140600030/29619259-white-rabbit-on-green-grass.jpg",
    "https://as2.ftcdn.net/v2/jpg/02/23/65/33/1000_F_223653386_8VRQmU0w8mNz8idEIaPW0scqFydUJhvH.jpg",
    "https://learnandsupport.getolympus.com/sites/default/files/styles/hero_large/public/2021-04/005_hopkins_ant19_23266.jpg?itok=rYGijQx0",
    "https://www.cottageonbunkerhill.com/wp-content/uploads/2021/08/30D2D6FB-1EC4-4502-8512-9A5A562F8B82-683x1024.jpeg",
    "https://www.kauveryhospitalsbangalore.com/assets/uploads/blog/mobcup_68526509d42eb.jpg",
    "https://images.pexels.com/photos/4397047/pexels-photo-4397047.jpeg",
    "https://cdn.justonecookbook.com/spai/q_glossy+ret_img+to_auto/www.justonecookbook.com/wp-content/uploads/2020/01/Sushi-Rolls-Maki-Sushi-%E2%80%93-Hosomaki-1106-II.jpg",
    "https://www.katiebirdbakes.com/wp-content/uploads/2018/02/Small-Chocolate-Cake-Ganache-katiebirdbakes-1-500x500.jpg",
    "https://thumbs.dreamstime.com/b/croissants-60018.jpg",
    "https://i.pinimg.com/736x/fa/0e/14/fa0e1497b9ea5b0837358643ea60da1b.jpg",
    "https://www.elmundoeats.com/wp-content/uploads/2021/02/FP-Quick-30-minutes-chicken-ramen-500x500.jpg",
    "https://thumbs.dreamstime.com/b/close-up-shot-colorful-macarons-stacked-white-plate-pastel-hues-pink-yellow-green-purple-create-appealing-374925521.jpg",
    "https://images.stockcake.com/public/8/4/6/846d2b6a-4e7f-4015-a6a6-34e4e681714d_large/snowy-mountain-peak-stockcake.jpg",
    "https://images.stockcake.com/public/9/a/9/9a9f3eea-c1fb-4b50-8891-3b15a1b1073a_large/sunset-tropical-beach-stockcake.jpg",
    "https://images.pexels.com/photos/20262378/pexels-photo-20262378/free-photo-of-waterfall-in-deep-forest.jpeg",
    "https://img.mathrubhumi.com/view/acePublic/alias/contentid/1i4ihi6ghpvo03s4hmq/3/sundarapandiapuram.webp?$p=f162f31&f=16:9&w=1080&q=0.8"
]

In [55]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

session = requests.Session()
session.headers.update(headers)

image_features_list = []
model.eval()

with torch.no_grad():
    for url in tqdm(image_urls, desc="Loading images"):
      try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        content_type = response.headers.get('Content-Type', '')
        if not content_type.startswith('image/'):
          print(f"Skipped (not image): {url}")
          continue
        image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
        image = inference_transform(image)
        image_features_list.append(image)
      except Exception as e:
        print(f"Failed to load: {url}")
        print(f"Error: {type(e).__name__} - {e}")
        continue

image_tensor = torch.stack(image_features_list).to(device)

Loading images: 100%|██████████| 20/20 [00:07<00:00,  2.56it/s]


In [56]:
with torch.no_grad():
  image_features = model.encode_image(image_tensor)
  txt_features = model.encode_text(input_ids, attention_mask)

similarity = image_features @ txt_features.T

In [57]:
best_caption_per_image = similarity.argmax(dim=1)

correct_cap = 0
for i, idx in enumerate(best_caption_per_image):
  print(f"Image {i} -> {captions[idx]}")
  if(i == idx):
    correct_cap+=1
print(f"{correct_cap}/{len(image_urls)}")

Image 0 -> a red apple on white background
Image 1 -> a golden retriever dog sitting
Image 2 -> a slice of chocolate cake
Image 3 -> a field of sunflowers
Image 4 -> a rabbit in green grass
Image 5 -> a brown bear in the forest
Image 6 -> a penguin standing on ice
Image 7 -> a butterfly on a flower
Image 8 -> a plate of sushi rolls
Image 9 -> a bowl of fresh strawberries
Image 10 -> a glass of orange juice
Image 11 -> a slice of chocolate cake
Image 12 -> a glass of orange juice
Image 13 -> a bowl of fresh strawberries
Image 14 -> a bowl of ramen noodles
Image 15 -> a plate of colorful macarons
Image 16 -> tropical beach at sunset
Image 17 -> tropical beach at sunset
Image 18 -> a waterfall in a green forest
Image 19 -> a butterfly on a flower
12/20


In [58]:
best_caption_per_text = similarity.argmax(dim=0)

correct_img = 0
for j, idx in enumerate(best_caption_per_text):
  print(f"Caption {captions[j]} -> Image {idx}")
  if j == idx:
    correct_img += 1
print(f"{correct_img}/{len(image_urls)}")

Caption a red apple on white background -> Image 0
Caption a golden retriever dog sitting -> Image 5
Caption a white cat on a windowsill -> Image 6
Caption a horse standing in a field -> Image 5
Caption a rabbit in green grass -> Image 4
Caption a brown bear in the forest -> Image 5
Caption a penguin standing on ice -> Image 6
Caption a butterfly on a flower -> Image 7
Caption a cup of black coffee -> Image 2
Caption a bowl of fresh strawberries -> Image 7
Caption a plate of sushi rolls -> Image 11
Caption a slice of chocolate cake -> Image 11
Caption a basket of croissants -> Image 11
Caption a glass of orange juice -> Image 12
Caption a bowl of ramen noodles -> Image 14
Caption a plate of colorful macarons -> Image 9
Caption snow covered mountain peak -> Image 16
Caption tropical beach at sunset -> Image 17
Caption a waterfall in a green forest -> Image 18
Caption a field of sunflowers -> Image 3
10/20
